# 4. Evaluate the two models

Same idea as training: one section for `rechtsgebieden`, one for `procedures`. Run them separately.

Each saved model reads the ruling text and can output several labels. A label is kept if its score is at least 0.5.

Two held-out sets, never used to change weights:

- `test.jsonl`: balanced sample, main number for the report
- `natural_test.jsonl`: natural mix of years and areas

**Micro-F1** pools every label decision (common labels dominate). **Macro-F1** averages one F1 per label (rare labels count equally).

Run Rechtsgebieden now (`../models/rg` is saved). Run Procedures only after `../models/pr` exists.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction,
)

DATA = Path("../data/processed")
RG_DIR = Path("../models/rg")
PR_DIR = Path("../models/pr")
MAX_LEN = 512
EVAL_BATCH = 8

print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("rg saved:", (RG_DIR / "model.safetensors").exists())
print("pr saved:", (PR_DIR / "model.safetensors").exists())

c:\Users\nlwen\Desktop\hbo-corp-justid\2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda: True
NVIDIA GeForce RTX 3050 Laptop GPU
rg saved: True
pr saved: True


Helpers: load rows, tokenize, turn logits into 0/1 labels, print Micro/Macro and the per-label table.

In [4]:
def load_task(path: Path, field: str, keep: list[str], drop_empty: bool) -> pd.DataFrame:
    frame = pd.read_json(path, lines=True)
    keep_set = set(keep)
    frame["target"] = frame[field].map(lambda xs: [x for x in xs if x in keep_set])
    if drop_empty:
        frame = frame[frame["target"].map(len) > 0]
    return frame[["ecli", "text", "target"]].reset_index(drop=True)


def encode(frame: pd.DataFrame, mlb: MultiLabelBinarizer, tokenizer) -> Dataset:
    labels = mlb.transform(frame["target"]).astype("float32")
    data = Dataset.from_dict({"text": frame["text"].tolist(), "labels": labels.tolist()})
    done = {"n": 0, "total": len(data)}

    def tok(batch):
        out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)
        done["n"] += len(batch["text"])
        if done["n"] % 2000 < len(batch["text"]) or done["n"] >= done["total"]:
            print(f"tokenize {done['n']}/{done['total']}", flush=True)
        return out

    return data.map(tok, batched=True)


def scores(pred: EvalPrediction) -> dict:
    logits = pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = 1.0 / (1.0 + np.exp(-np.asarray(logits)))
    y_hat = (probs >= 0.5).astype(int)
    y = np.asarray(pred.label_ids)
    return {
        "f1_micro": f1_score(y, y_hat, average="micro", zero_division=0),
        "f1_macro": f1_score(y, y_hat, average="macro", zero_division=0),
        "precision_micro": precision_score(y, y_hat, average="micro", zero_division=0),
        "recall_micro": recall_score(y, y_hat, average="micro", zero_division=0),
    }


def make_trainer(model_dir: Path):
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    args = TrainingArguments(
        output_dir=str(model_dir / "_eval_scratch"),
        per_device_eval_batch_size=EVAL_BATCH,
        report_to=[],
        disable_tqdm=True,
        fp16=torch.cuda.is_available(),
    )
    return Trainer(model=model, args=args, compute_metrics=scores)


def run_split(name: str, frame: pd.DataFrame, mlb: MultiLabelBinarizer, tokenizer, trainer) -> dict:
    print(f"\n=== {name}: {len(frame)} rows ===")
    ds = encode(frame, mlb, tokenizer)
    out = trainer.predict(ds)
    metrics = dict(out.metrics)
    for key in ("test_f1_micro", "test_f1_macro", "test_precision_micro", "test_recall_micro"):
        if key in metrics:
            print(f"{key}: {metrics[key]:.4f}")
    logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    probs = 1.0 / (1.0 + np.exp(-np.asarray(logits)))
    y_hat = (probs >= 0.5).astype(int)
    y = np.asarray(out.label_ids)
    print(classification_report(y, y_hat, target_names=list(mlb.classes_), zero_division=0))
    return {"metrics": metrics, "y_hat": y_hat, "y": y}

## Rechtsgebieden

Load `../models/rg`. Run this block now.

In [3]:
rg_keep = json.loads((RG_DIR / "labels.json").read_text(encoding="utf-8"))
rg_tokenizer = AutoTokenizer.from_pretrained(RG_DIR)
rg_mlb = MultiLabelBinarizer(classes=rg_keep)
rg_mlb.fit([rg_keep])
rg_trainer = make_trainer(RG_DIR)

rg_test = load_task(DATA / "test.jsonl", "rechtsgebieden", rg_keep, drop_empty=True)
rg_natural = load_task(DATA / "natural_test.jsonl", "rechtsgebieden", rg_keep, drop_empty=True)
print("RG test:", len(rg_test), "natural_test:", len(rg_natural))
print("labels:", len(rg_keep))

RG test: 6068 natural_test: 4985
labels: 20


Score the balanced test set, then the natural set. Each split prints Micro/Macro and one row per label.

In [4]:
rg_test_out = run_split("RG test", rg_test, rg_mlb, rg_tokenizer, rg_trainer)
rg_nat_out = run_split("RG natural_test", rg_natural, rg_mlb, rg_tokenizer, rg_trainer)

rg_eval = {
    "test": {k: float(v) for k, v in rg_test_out["metrics"].items() if k.startswith("test_")},
    "natural_test": {k: float(v) for k, v in rg_nat_out["metrics"].items() if k.startswith("test_")},
}
(RG_DIR / "eval.json").write_text(json.dumps(rg_eval, indent=2), encoding="utf-8")
print("wrote", RG_DIR / "eval.json")


=== RG test: 6068 rows ===


Map:  16%|█▋        | 1000/6068 [00:02<00:11, 435.07 examples/s]

tokenize 2000/6068


Map:  49%|████▉     | 3000/6068 [00:08<00:08, 345.90 examples/s]

tokenize 4000/6068


...

tokenize 4000/4985


Map:  80%|████████  | 4000/4985 [00:17<00:04, 211.68 examples/s]

tokenize 4985/4985


Map: 100%|██████████| 4985/4985 [00:20<00:00, 244.34 examples/s]


test_f1_micro: 0.9613
test_f1_macro: 0.7476
test_precision_micro: 0.9640
test_recall_micro: 0.9587
                             precision    recall  f1-score   support

                 Strafrecht       0.99      0.99      0.99      1051
              Bestuursrecht       1.00      1.00      1.00      2391
               Civiel recht       1.00      0.98      0.99      1573
     Socialezekerheidsrecht       0.94      0.96      0.95       519
  Personen- en familierecht       0.89      0.92      0.91       312
             Belastingrecht       0.98      1.00      0.99       436
         Vreemdelingenrecht       0.97      0.99      0.98       461
        Verbintenissenrecht       0.65      0.64      0.64       164
             Omgevingsrecht       0.74      0.80      0.76       108
            Ambtenarenrecht       0.91      0.89      0.90        79
           Insolventierecht       0.91      0.84      0.87        57
               Arbeidsrecht       0.76      0.69      0.72        70
Int

A few test rows: gold labels vs predicted labels.

In [5]:
names = list(rg_mlb.classes_)
for i in range(8):
    gold = [n for n, on in zip(names, rg_test_out["y"][i]) if on]
    pred = [n for n, on in zip(names, rg_test_out["y_hat"][i]) if on]
    print(rg_test.loc[i, "ecli"])
    print("  gold:", gold)
    print("  pred:", pred)

ECLI:NL:ORBBACM:2015:22
  gold: ['Bestuursrecht', 'Belastingrecht']
  pred: ['Bestuursrecht', 'Belastingrecht']
ECLI:NL:RBDHA:2015:8473
  gold: ['Bestuursrecht', 'Vreemdelingenrecht']
  pred: ['Bestuursrecht', 'Vreemdelingenrecht']
ECLI:NL:RBGEL:2023:7049
  gold: ['Strafrecht']
  pred: ['Strafrecht']
ECLI:NL:RBAMS:2020:4331
  gold: ['Strafrecht', 'Internationaal publiekrecht']
  pred: ['Strafrecht', 'Internationaal publiekrecht']
ECLI:NL:OGEAM:2016:64
  gold: ['Civiel recht']
  pred: ['Civiel recht']
ECLI:NL:RBNNE:2019:4872
  gold: ['Strafrecht']
  pred: ['Strafrecht']
ECLI:NL:RBZWB:2024:4999
  gold: ['Civiel recht']
  pred: ['Civiel recht', 'Verbintenissenrecht']
ECLI:NL:RBGEL:2013:6181
  gold: ['Strafrecht']
  pred: ['Strafrecht']


## Procedures

Same checks. Run this block only after `03_train` has saved `../models/pr`.

In [5]:
if not (PR_DIR / "model.safetensors").exists():
    raise FileNotFoundError("../models/pr is not saved yet. Train procedures first, then run these cells.")

pr_keep = json.loads((PR_DIR / "labels.json").read_text(encoding="utf-8"))
pr_tokenizer = AutoTokenizer.from_pretrained(PR_DIR)
pr_mlb = MultiLabelBinarizer(classes=pr_keep)
pr_mlb.fit([pr_keep])
pr_trainer = make_trainer(PR_DIR)

pr_test = load_task(DATA / "test.jsonl", "procedures", pr_keep, drop_empty=True)
pr_natural = load_task(DATA / "natural_test.jsonl", "procedures", pr_keep, drop_empty=True)
print("PR test:", len(pr_test), "natural_test:", len(pr_natural))
print("labels:", len(pr_keep))

PR test: 5723 natural_test: 4791
labels: 25


In [6]:
pr_test_out = run_split("PR test", pr_test, pr_mlb, pr_tokenizer, pr_trainer)
pr_nat_out = run_split("PR natural_test", pr_natural, pr_mlb, pr_tokenizer, pr_trainer)

pr_eval = {
    "test": {k: float(v) for k, v in pr_test_out["metrics"].items() if k.startswith("test_")},
    "natural_test": {k: float(v) for k, v in pr_nat_out["metrics"].items() if k.startswith("test_")},
}
(PR_DIR / "eval.json").write_text(json.dumps(pr_eval, indent=2), encoding="utf-8")
print("wrote", PR_DIR / "eval.json")


=== PR test: 5723 rows ===


Map:  17%|█▋        | 1000/5723 [00:02<00:13, 353.27 examples/s]

tokenize 2000/5723


Map:  52%|█████▏    | 3000/5723 [00:09<00:09, 299.12 examples/s]

tokenize 4000/5723


...

tokenize 4000/4791


Map:  83%|████████▎ | 4000/4791 [00:15<00:03, 241.50 examples/s]

tokenize 4791/4791


Map: 100%|██████████| 4791/4791 [00:18<00:00, 258.04 examples/s]


test_f1_micro: 0.9260
test_f1_macro: 0.7650
test_precision_micro: 0.9344
test_recall_micro: 0.9178
                                  precision    recall  f1-score   support

                    Hoger beroep       0.99      0.98      0.99      1535
      Eerste aanleg - meervoudig       0.98      0.94      0.96       864
     Eerste aanleg - enkelvoudig       0.92      0.95      0.93      1393
                        Cassatie       1.00      1.00      1.00       205
                  Op tegenspraak       0.86      0.70      0.77       175
                     Kort geding       0.91      0.97      0.94       177
                     Beschikking       0.85      0.84      0.84       190
          Voorlopige voorziening       0.92      0.89      0.90       200
                       Bodemzaak       0.77      0.60      0.68       101
          Eerste en enige aanleg       0.89      0.71      0.79        34
                 Rekestprocedure       0.71      0.71      0.71        41
            

In [ ]:
names = list(pr_mlb.classes_)
for i in range(8):
    gold = [n for n, on in zip(names, pr_test_out["y"][i]) if on]
    pred = [n for n, on in zip(names, pr_test_out["y_hat"][i]) if on]
    print(pr_test.loc[i, "ecli"])
    print("  gold:", gold)
    print("  pred:", pred)

ECLI:NL:ORBBACM:2015:22
  gold: ['Hoger beroep']
  pred: ['Beschikking']
ECLI:NL:RBDHA:2015:8473
  gold: ['Voorlopige voorziening']
  pred: ['Voorlopige voorziening']
ECLI:NL:RBGEL:2023:7049
  gold: ['Eerste aanleg - meervoudig', 'Op tegenspraak']
  pred: ['Eerste aanleg - meervoudig']
ECLI:NL:RBAMS:2020:4331
  gold: ['Op tegenspraak', 'Eerste en enige aanleg']
  pred: ['Eerste en enige aanleg']
ECLI:NL:OGEAM:2016:64
  gold: ['Eerste aanleg - enkelvoudig']
  pred: ['Eerste aanleg - enkelvoudig']
ECLI:NL:RBNNE:2019:4872
  gold: ['Eerste aanleg - meervoudig', 'Op tegenspraak']
  pred: ['Eerste aanleg - meervoudig', 'Op tegenspraak']
ECLI:NL:RBZWB:2024:4999
  gold: ['Bodemzaak']
  pred: ['Bodemzaak']
ECLI:NL:RBGEL:2013:6181
  gold: ['Eerste aanleg - meervoudig']
  pred: ['Eerste aanleg - meervoudig']


: 